# Assignment #02

by Patrick Donnelly & Burke Havranek

EECE 6544: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

A telecom company ("Telco") sells a base mobile plan and lets customers bolt on optional add-on services phone
service, multiple lines, fiber vs. DSL internet, online security, online backup, device protection, tech support,
streaming TV, and streaming movies. Each add-on nudges the monthly bill upward, but not by the same amount
a fiber-internet upgrade costs far more than device protection.

**The billing and revenue team wants to answer a practical question:**

**Given *which* add-on services a customer subscribes to, what monthly charge should we expect them to pay?**

## Part 1: Data Analysis

### Section 1: Loading and Inspecting the Data

In [1]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv("telco.csv")
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,target,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,0,0,0,1,29.85,29.85,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,0,0,0,0,56.95,1889.5,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,0,0,0,1,53.85,108.15,1,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,1,0,0,0,42.30,1840.75,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,0,0,0,0,0,0,0,1,70.70,151.65,1,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,1,1,0,1,1,1,1,1,84.80,1990.5,0,0,0,1,0,0,0,1
7039,0,0,1,1,72,1,1,0,1,1,0,1,1,1,103.20,7362.9,0,1,0,1,0,1,0,0
7040,0,0,1,1,11,0,0,1,0,0,0,0,0,1,29.60,346.45,0,0,0,0,0,0,1,0
7041,1,1,1,0,4,1,1,0,0,0,0,0,0,1,74.40,306.6,1,1,0,0,0,0,0,1


In the above `telco.csv` file, we find 7043 entries of 24 fields, as expected, with the `MonthlyCharges` field from the original set having been renamed to `target`, also as expected. Looking for missing entries, we find

In [2]:
df.isnull().sum()

gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
target                                   0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_Electronic check           0
PaymentMeth

In [3]:
df.isna().sum()

gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
target                                   0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_Electronic check           0
PaymentMeth

Thus, no entries are missing. To verify all service columns are strictly Boolean, we find

In [4]:
SERVICES = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", \
            "StreamingTV", "StreamingMovies", "InternetService_Fiber optic", "InternetService_No"]

df[~df[SERVICES].isin([0,1])].count()

gender                                   0
SeniorCitizen                            0
Partner                                  0
Dependents                               0
tenure                                   0
PhoneService                             0
MultipleLines                            0
OnlineSecurity                           0
OnlineBackup                             0
DeviceProtection                         0
TechSupport                              0
StreamingTV                              0
StreamingMovies                          0
PaperlessBilling                         0
target                                   0
TotalCharges                             0
Churn                                    0
InternetService_Fiber optic              0
InternetService_No                       0
Contract_One year                        0
Contract_Two year                        0
PaymentMethod_Credit card (automatic)    0
PaymentMethod_Electronic check           0
PaymentMeth

Ergo, all fields of interest are populated and sanitized properly.

### Section 2: Variable Definition

We wish to perform a simple multivariable linear regression with the above 10 `SERVICES` fields as independent variables and the `target` field, representing monthly charges, as the dependent variable. Dividing these terms into separate variables, we find

In [5]:
X = df[SERVICES]
X

,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,InternetService_Fiber optic,InternetService_No
0,0,0,0,1,0,0,0,0,0,0
1,1,0,1,0,1,0,0,0,0,0
2,1,0,1,1,0,0,0,0,0,0
3,0,0,1,0,1,1,0,0,0,0
4,1,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...
7038,1,1,1,0,1,1,1,1,0,0
7039,1,1,0,1,1,0,1,1,1,0
7040,0,0,1,0,0,0,0,0,0,0
7041,1,1,0,0,0,0,0,0,1,0


In [6]:
y = df['target']
y

0        29.85
1        56.95
2        53.85
3        42.30
4        70.70
         ...  
7038     84.80
7039    103.20
7040     29.60
7041     74.40
7042    105.65
Name: target, Length: 7043, dtype: float64

### Section 3: Data Splitting

Before we may train any machine learning model on this data, we must first divide into a training set and a test set. Using a standard 80/20 split, we find

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.2, random_state=0xDEADBEEF)

### Section 4: Multivariable Linear Regression

Given our above split data, we therefore train a linear regression model by

In [8]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

### Section 5: Parameter Report

The above applet contains, *inter alia*, the parameters of our linear regression model. Of interest however are the coefficients `m` and bias `b`, given by

In [9]:
m = model.coef_
m

array([ 20.05386194,   5.01662829,   5.02717226,   4.97621893,
         5.03237593,   5.06092672,   9.98778679,   9.97487358,
        24.93551074, -25.0272209 ])

In [10]:
b = model.intercept_
b

24.936592371336232


### Section 6: Model Evaluation


To evaluate the model, R2 score, mean absolute error, root mean squared error, and the mean square average can be used.

In [11]:
from sklearn.metrics import r2_score

y_pred = model.predict(X_test)

print("R^2 :", r2_score(y_test, y_pred))

R^2 : 0.9988553887253341


In [12]:
from sklearn.metrics import mean_absolute_error

print("MAE:", mean_absolute_error(y_test, y_pred))

MAE: 0.7798171319835341


In [13]:
from sklearn.metrics import root_mean_squared_error

print("RMSE:", root_mean_squared_error(y_test, y_pred))

RMSE: 1.0115734288374625


In [14]:
from sklearn.metrics import mean_squared_error

print("MSA:", mean_squared_error(y_test, y_pred))

MSA: 1.0232808019299808


Looking at the model, The R2 at 99.9% which is very accurate. The other evaluation metrics are low which is ideal.


### Section 7: Baseline Comparison


To see the baseline comparison between a multi variable and a single variable model a single variable model needs to be made, the model with only one coefficient will be `my_model_single_coef` and will be used to compare the differences in evaluation between multi and single coefficient linear regression models for this data.

In [15]:
# need to train a single variable model

df_single_coef = pd.read_csv("telco.csv")

In [16]:
addon_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies']

df_single_coef_2 = pd.DataFrame({
    'num_addons': df_single_coef[addon_cols].sum(axis=1),   # sum across the row = how many add-ons
    'monthly_charge': df_single_coef['target']              # 'target' holds the monthly charge
})

df_single_coef_2['num_addons'].max()


X_single_coef = df_single_coef_2[["num_addons"]]   # feature — DataFrame (9 rows × 1 col)
Y_single_coef = df_single_coef_2["monthly_charge"]      # target — Series

Y_single_coef

0        29.85
1        56.95
2        53.85
3        42.30
4        70.70
         ...  
7038     84.80
7039    103.20
7040     29.60
7041     74.40
7042    105.65
Name: monthly_charge, Length: 7043, dtype: float64

In [17]:

X_train_single, X_test_single, y_train_single, y_test_single = \
    train_test_split(X_single_coef, Y_single_coef, test_size=0.2, random_state=0xDEADBEEF)

my_model_single_coef=LinearRegression()
my_model_single_coef.fit(X_single_coef,Y_single_coef)

single_coef = my_model_single_coef.coef_
single_coef

single_intercept = my_model_single_coef.intercept_
single_intercept

y_pred_single = my_model_single_coef.predict(X_test_single)

print("MAE for the single coef :", mean_absolute_error(y_test_single, y_pred_single))
print("R^2 for the single coef :", r2_score(y_test_single, y_pred_single))
print("RMSE for the single coef :", root_mean_squared_error(y_test_single, y_pred_single))
print("MSA for the single coef :", mean_squared_error(y_test_single, y_pred_single))


MAE for the single coef : 17.046274288690178
R^2 for the single coef : 0.5866539042714232
RMSE for the single coef : 19.223184017033986
MSA for the single coef : 369.5308037527509


The single variable model has a 58.7% R^2 score, significantly lower then the multi variable `model`. The other metrics show that the `my_model_single_coef` model does not have as accurate predictions as well


### Section 8: Results


In [18]:
base_busniess_cost = model.intercept_

print_out_loop = 0

for  print_out_loop in range(m.shape[0]):
    print(f"The monthly service cost of {SERVICES[print_out_loop]} : ${m[print_out_loop]:.2f}")

The monthly service cost of PhoneService : $20.05
The monthly service cost of MultipleLines : $5.02
The monthly service cost of OnlineSecurity : $5.03
The monthly service cost of OnlineBackup : $4.98
The monthly service cost of DeviceProtection : $5.03
The monthly service cost of TechSupport : $5.06
The monthly service cost of StreamingTV : $9.99
The monthly service cost of StreamingMovies : $9.97
The monthly service cost of InternetService_Fiber optic : $24.94
The monthly service cost of InternetService_No : $-25.03


In the context of the above, all services appear to have a rather straightforward monthly cost of approximately $5, $10, or $20, with a lack of internet service translating to a reimbursement of approximately $25, compared to a base fiber optic internet service cost of approximately $25.

### Section 9: Predictions


This `test_customer` can help see how the `model` can predict costs for a customer

In [19]:
test_customer = [1,1,0,0,0,0,0,0,1,0] # may need vaild feature names

final_prediction = model.predict([test_customer]) #not sure on what makes a good test prediction base

final_prediction

c:\Users\burke\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([74.94259335])

Hence, given a standard customer (base cost ~$25) with multi-line phone service (cost approximately $20 + $5, respectively) and fiber optic internet (cost ~$25), we calculate an approximate total cost above of ~$75, as expected.

## Part 2: Practical Considerations


### Section 1: Base Cost

Looking at the intercept represents the base price of a deal with Telco. The base cost is a approximately $24.95.


### Section 2: Add-On Contributions

The coefficient represents the cost of each add on. Each service added on cost can be seen below:

The monthly service cost of `PhoneService` : $20.05 / mo

The monthly service cost of `MultipleLines` : $5.01 / mo

The monthly service cost of `OnlineSecurity` : $5.01 / mo

The monthly service cost of `OnlineBackup` : $4.99 / mo

The monthly service cost of `DeviceProtection` : $5.02 / mo

The monthly service cost of `TechSupport` : $5.03 / mo

The monthly service cost of `StreamingTV` : $9.97 / mo

The monthly service cost of `StreamingMovies` : $9.96 / mo

The monthly service cost of `InternetService_Fiber` optic : $24.96 / mo

The monthly service cost of `InternetService_No` : -$25.05 / mo


### Section 3: Min And Max Add-Ons

Looking at the maximum and minimum, the Fiber optic service is the most expensive add-on and Multiple lines, and online security are the cheapest add-ons. The -$25.05 add-on for no internet services is not an add-on in the same way it is a removal of services.


### Section 4: Price Increases

If someone were to add both streaming services and fiber, it would be the equivalent of adding those three coefficients together:

$24.96+ $9.96+ $9.97 = $44.89

It would result in a $44.89 increase to the monthly bill if none are previously  included.


### Section 5: Insight From Predations

The prediction tested was for someone with phone services, multiple lines and fiber optic. The expected monthly cost of that possible customer would be $74.97


### Section 6: Single Input Versus Specific

Comparing the single coefficient model to the multi-variable model can give insight into having more or less inputs into this data set. Looking at the evaluation tests on both models, the multi-value model performed significantly better. This would indicate that the specific variables are much better than knowing simply the total of services.


### Section 7: Evaluation Analysis

The accuracy of the predictions can be shown by the mean absolute error. The mean absolute error of the multivariable model is $0.78 off of the actual monthly expenses.


### Section 8: Outliers and Real-World Values

The "internet services no" option is the only option that is in negative. This also throws off the single coefficient model. Fiber optic seems to be the biggest monthly charge after the base cost. Other than this, all calculated values are in line with typical telco prices.
